# Week 07 · Tuesday — Word2Vec & Semantic Similarity
NLP Foundations · ShopSense Reviews Dataset

In [7]:
import re
import numpy as np
import pandas as pd
from collections import defaultdict
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

C:\Users\abeku\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
REVIEWS_PATH  = "shopsense_reviews.csv"
W2V_VECTOR_SIZE = 100
W2V_EPOCHS      = 15
W2V_MIN_COUNT   = 1
SEED            = 42

# anchor words that exist in the ShopSense vocab
CHEAP_POSITIVE_ANCHORS  = ["price", "value", "worth"]   # 'affordable' sense
CHEAP_NEGATIVE_ANCHORS  = ["poor", "material", "terrible"]  # 'low-quality' sense

REVIEW_A = "incredible camera but terrible battery life"
REVIEW_B = "Battery drains fast, although photos are stunning"

In [9]:
def load_reviews(path):
    """Load reviews CSV and validate expected columns."""
    try:
        df = pd.read_csv(path)
        assert 'review_text' in df.columns, "Missing review_text"
        assert 'category'    in df.columns, "Missing category"
        df['review_text'] = df['review_text'].fillna("")
        print(f"Loaded {len(df)} reviews — categories: {df['category'].unique().tolist()}")
        return df
    except FileNotFoundError as e:
        print(f"File not found: {e}")
        raise


def tokenize(text):
    """Lowercase alphabetic tokenisation — mirrors Monday's approach."""
    return re.findall(r'[a-z]+', text.lower())


reviews_df = load_reviews(REVIEWS_PATH)
corpus_tokens = [tokenize(t) for t in reviews_df['review_text']]

Loaded 10199 reviews — categories: ['Clothing', 'Home', 'Electronics', 'Books', 'Beauty', 'Food']


---
## Q1 — Word2Vec on ShopSense Reviews

### (a) 'cheap' is polysemous — one vector problem

In [10]:
def train_word2vec(corpus_tokens, vector_size, window, min_count, epochs, seed):
    """Train a Word2Vec skip-gram model and return it."""
    model = Word2Vec(
        sentences=corpus_tokens,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        sg=1,          # skip-gram
        workers=2,
        epochs=epochs,
        seed=seed
    )
    return model


w2v_model = train_word2vec(corpus_tokens, W2V_VECTOR_SIZE, window=5,
                           min_count=W2V_MIN_COUNT, epochs=W2V_EPOCHS, seed=SEED)

print(f"Vocabulary size: {len(w2v_model.wv)}")
print(f"'cheap' in vocab: {'cheap' in w2v_model.wv}")
print(f"Vector for 'cheap' (first 10 dims): {w2v_model.wv['cheap'][:10]}")
print()
print("Top-10 nearest neighbours to 'cheap':")
for word, sim in w2v_model.wv.most_similar('cheap', topn=10):
    print(f"  {word:20s} {sim:.4f}")

Vocabulary size: 255
'cheap' in vocab: True
Vector for 'cheap' (first 10 dims): [-0.3533716   0.1270265  -0.44904715 -0.86304617  0.11706256 -0.8449544
  0.35884205  0.73071545 -0.76489884 -0.5712859 ]

Top-10 nearest neighbours to 'cheap':
  material             0.9692
  finishing            0.9089
  much                 0.7615
  price                0.6871
  poor                 0.6737
  worth                0.6265
  thoda                0.5731
  better               0.5503
  zyada                0.5489
  expected             0.5483


In [11]:
def cosine_pair(model, word_a, word_b):
    """Cosine similarity between two words. Returns None if either is OOV."""
    if word_a not in model.wv or word_b not in model.wv:
        return None
    return float(model.wv.similarity(word_a, word_b))


# 'affordable' / 'flimsy' are not in this corpus — use in-vocab proxy words
# positive (affordable) sense proxies: price, value, worth
# negative (low-quality) sense proxies: poor, material, terrible
positive_proxies = CHEAP_POSITIVE_ANCHORS
negative_proxies = CHEAP_NEGATIVE_ANCHORS

print("cosine('cheap', positive-sense proxies) — affordable meaning:")
for w in positive_proxies:
    sim = cosine_pair(w2v_model, 'cheap', w)
    print(f"  cosine('cheap', '{w}') = {sim:.4f}")

print()
print("cosine('cheap', negative-sense proxies) — low-quality meaning:")
for w in negative_proxies:
    sim = cosine_pair(w2v_model, 'cheap', w)
    print(f"  cosine('cheap', '{w}') = {sim:.4f}")

print("""
The polysemy problem:
Word2Vec assigns exactly ONE 300-d (here 100-d) point to 'cheap',
averaging over all its usages in the corpus — both 'cheap price' (good)
and 'cheap material' (bad). Contextualised models like BERT generate
a different vector per occurrence depending on surrounding tokens.
""")

cosine('cheap', positive-sense proxies) — affordable meaning:
  cosine('cheap', 'price') = 0.6871
  cosine('cheap', 'value') = 0.3461
  cosine('cheap', 'worth') = 0.6265

cosine('cheap', negative-sense proxies) — low-quality meaning:
  cosine('cheap', 'poor') = 0.6737
  cosine('cheap', 'material') = 0.9692
  cosine('cheap', 'terrible') = 0.2937

The polysemy problem:
Word2Vec assigns exactly ONE 300-d (here 100-d) point to 'cheap',
averaging over all its usages in the corpus — both 'cheap price' (good)
and 'cheap material' (bad). Contextualised models like BERT generate
a different vector per occurrence depending on surrounding tokens.



### (b) Disambiguation system — 'cheap' sense from context

In [12]:
def get_anchor_vector(model, anchor_words):
    """
    Compute a centroid anchor vector from a list of seed words.
    Words not in vocab are silently skipped.
    """
    vecs = [model.wv[w] for w in anchor_words if w in model.wv]
    if not vecs:
        raise ValueError(f"None of {anchor_words} found in vocabulary.")
    return np.mean(vecs, axis=0)


def context_vector(model, sentence, target_word):
    """
    Average embedding of all words in sentence EXCEPT the target.
    Represents the local context signal.
    """
    tokens = tokenize(sentence)
    context = [t for t in tokens if t != target_word and t in model.wv]
    if not context:
        raise ValueError(f"No context words found in vocab for sentence: {sentence}")
    return np.mean([model.wv[w] for w in context], axis=0)


def disambiguate_cheap(model, sentence, target="cheap",
                       positive_anchors=CHEAP_POSITIVE_ANCHORS,
                       negative_anchors=CHEAP_NEGATIVE_ANCHORS):
    """
    Determine if 'cheap' in the sentence means 'affordable' or 'low-quality'
    by comparing the context vector to two anchor centroids.
    """
    if target not in tokenize(sentence):
        return f"'{target}' not found in sentence."

    ctx_vec   = context_vector(model, sentence, target)
    pos_anchor = get_anchor_vector(model, positive_anchors)
    neg_anchor = get_anchor_vector(model, negative_anchors)

    sim_pos = float(cosine_similarity([ctx_vec], [pos_anchor])[0][0])
    sim_neg = float(cosine_similarity([ctx_vec], [neg_anchor])[0][0])

    sense = "affordable (positive)" if sim_pos > sim_neg else "low-quality (negative)"
    print(f"Sentence : {sentence}")
    print(f"  sim(context, affordable-anchor) = {sim_pos:.4f}")
    print(f"  sim(context, low-quality-anchor) = {sim_neg:.4f}")
    print(f"  => Predicted sense: {sense}")
    return sense


test_sentences = [
    "this product is cheap and great value for money",
    "the material is cheap and broke after one week",
    "bought at a cheap price worth every rupee",
    "terrible quality cheap product waste of money",
]

print("=== Cheap Disambiguation ===")
for s in test_sentences:
    disambiguate_cheap(w2v_model, s)
    print()

=== Cheap Disambiguation ===
Sentence : this product is cheap and great value for money
  sim(context, affordable-anchor) = 0.6286
  sim(context, low-quality-anchor) = 0.5337
  => Predicted sense: affordable (positive)

Sentence : the material is cheap and broke after one week
  sim(context, affordable-anchor) = 0.5522
  sim(context, low-quality-anchor) = 0.6809
  => Predicted sense: low-quality (negative)

Sentence : bought at a cheap price worth every rupee
  sim(context, affordable-anchor) = 0.8691
  sim(context, low-quality-anchor) = 0.6246
  => Predicted sense: affordable (positive)

Sentence : terrible quality cheap product waste of money
  sim(context, affordable-anchor) = 0.4686
  sim(context, low-quality-anchor) = 0.7082
  => Predicted sense: low-quality (negative)



### (c) window=2 vs window=10 — syntactic vs semantic

In [13]:
def compare_windows(corpus_tokens, probe_word, vector_size, epochs, seed):
    """
    Train two models with small vs large window and compare nearest neighbours.
    """
    model_narrow = train_word2vec(corpus_tokens, vector_size, window=2,
                                  min_count=1, epochs=epochs, seed=seed)
    model_wide   = train_word2vec(corpus_tokens, vector_size, window=10,
                                  min_count=1, epochs=epochs, seed=seed)

    if probe_word not in model_narrow.wv:
        print(f"'{probe_word}' not in vocab.")
        return

    print(f"Nearest neighbours to '{probe_word}'")
    print(f"{'window=2 (syntactic)':35s}  {'window=10 (semantic)':35s}")
    print("-" * 75)
    narrow_nn = model_narrow.wv.most_similar(probe_word, topn=8)
    wide_nn   = model_wide.wv.most_similar(probe_word, topn=8)
    for (w2, s2), (w10, s10) in zip(narrow_nn, wide_nn):
        print(f"  {w2:20s} {s2:.4f}        {w10:20s} {s10:.4f}")

    return model_narrow, model_wide


model_w2, model_w10 = compare_windows(corpus_tokens, 'quality',
                                       W2V_VECTOR_SIZE, W2V_EPOCHS, SEED)

print("""
Interpretation:
window=2 forces the model to learn from immediately adjacent tokens,
so neighbours tend to be syntactically similar (same POS, co-occurrence
patterns like adjective-noun). window=10 sweeps a much broader context,
capturing topically related words that appear in the same review even
without being adjacent — that's the semantic relationship.
For sentiment tasks, window=5 is typically the sweet spot.
""")

Nearest neighbours to 'quality'
window=2 (syntactic)                 window=10 (semantic)               
---------------------------------------------------------------------------
  satisfied            0.5125        satisfied            0.3914
  thoda                0.5031        sure                 0.3713
  excellent            0.4561        price                0.3628
  finishing            0.4358        again                0.3524
  zyada                0.4275        is                   0.3432
  highly               0.4147        poor                 0.3362
  bohot                0.4130        but                  0.3116
  friends              0.3986        buy                  0.3090

Interpretation:
window=2 forces the model to learn from immediately adjacent tokens,
so neighbours tend to be syntactically similar (same POS, co-occurrence
patterns like adjective-noun). window=10 sweeps a much broader context,
capturing topically related words that appear in the same review even

---
## Q2 — Four Similarity Methods on Reviews A & B

In [14]:
review_a = REVIEW_A
review_b = REVIEW_B
print(f"Review A: {review_a}")
print(f"Review B: {review_b}")
print()
# Both express mixed sentiment: good camera/photos, bad battery

Review A: incredible camera but terrible battery life
Review B: Battery drains fast, although photos are stunning



### Method 1 — Bag of Words

In [15]:
def bow_similarity(text_a, text_b):
    """Cosine similarity using binary bag-of-words vectors."""
    vectorizer = CountVectorizer(binary=True)
    mat = vectorizer.fit_transform([text_a, text_b])
    sim = float(cosine_similarity(mat[0], mat[1])[0][0])

    tokens_a = set(tokenize(text_a))
    tokens_b = set(tokenize(text_b))
    overlap   = tokens_a & tokens_b
    union     = tokens_a | tokens_b

    print("=== BOW ===")
    print(f"  Tokens A   : {sorted(tokens_a)}")
    print(f"  Tokens B   : {sorted(tokens_b)}")
    print(f"  Overlap    : {sorted(overlap)} ({len(overlap)} words)")
    print(f"  Cosine sim : {sim:.4f}")
    return sim


sim_bow = bow_similarity(review_a, review_b)

=== BOW ===
  Tokens A   : ['battery', 'but', 'camera', 'incredible', 'life', 'terrible']
  Tokens B   : ['although', 'are', 'battery', 'drains', 'fast', 'photos', 'stunning']
  Overlap    : ['battery'] (1 words)
  Cosine sim : 0.1543


### Method 2 — TF-IDF

In [16]:
def tfidf_similarity(text_a, text_b, background_corpus=None):
    """
    Cosine similarity using TF-IDF. If background_corpus is provided,
    IDF is estimated from it for more realistic weighting.
    """
    vectorizer = TfidfVectorizer(token_pattern=r'[a-z]+')
    if background_corpus:
        vectorizer.fit(background_corpus)
        mat = vectorizer.transform([text_a, text_b])
    else:
        mat = vectorizer.fit_transform([text_a, text_b])

    sim = float(cosine_similarity(mat[0], mat[1])[0][0])
    print(f"=== TF-IDF === cosine: {sim:.4f}")
    return sim


background = reviews_df['review_text'].tolist()
sim_tfidf = tfidf_similarity(review_a, review_b, background_corpus=background)

=== TF-IDF === cosine: 0.4202


### Method 3 — Word2Vec averaging

In [17]:
def w2v_avg_similarity(model, text_a, text_b):
    """
    Represent each document as the mean of its in-vocab word vectors,
    then compute cosine similarity.
    """
    def doc_vector(text):
        tokens = [t for t in tokenize(text) if t in model.wv]
        if not tokens:
            return np.zeros(model.vector_size)
        return np.mean([model.wv[t] for t in tokens], axis=0)

    vec_a = doc_vector(text_a)
    vec_b = doc_vector(text_b)

    sim = float(cosine_similarity([vec_a], [vec_b])[0][0])

    in_vocab_a = [t for t in tokenize(text_a) if t in model.wv]
    in_vocab_b = [t for t in tokenize(text_b) if t in model.wv]
    print(f"=== Word2Vec avg ===")
    print(f"  In-vocab tokens A: {in_vocab_a}")
    print(f"  In-vocab tokens B: {in_vocab_b}")
    print(f"  Cosine sim: {sim:.4f}")
    return sim


sim_w2v = w2v_avg_similarity(w2v_model, review_a, review_b)

=== Word2Vec avg ===
  In-vocab tokens A: ['camera', 'but', 'terrible', 'battery']
  In-vocab tokens B: ['battery', 'fast', 'are']
  Cosine sim: 0.7169


### Method 4 — Sentence-BERT

In [18]:
def sbert_similarity(text_a, text_b, model_name="all-MiniLM-L6-v2"):
    """Encode sentences with Sentence-BERT and compute cosine similarity."""
    try:
        sbert = SentenceTransformer(model_name)
        emb_a, emb_b = sbert.encode([text_a, text_b])
        sim = float(cosine_similarity([emb_a], [emb_b])[0][0])
        print(f"=== Sentence-BERT ({model_name}) ===")
        print(f"  Embedding dim: {len(emb_a)}")
        print(f"  Cosine sim   : {sim:.4f}")
        return sim
    except Exception as e:
        print(f"SBERT error: {e}")
        return None


sim_sbert = sbert_similarity(review_a, review_b)

C:\Users\abeku\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\abeku\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3961.68it/s]
BertMo

=== Sentence-BERT (all-MiniLM-L6-v2) ===
  Embedding dim: 384
  Cosine sim   : 0.6336


### (a) Which method correctly identifies similarity?

In [19]:
def summarise_methods(scores: dict):
    """Print a comparison table of similarity scores."""
    print(f"{'Method':25s} {'Score':>8s}  Captures semantic similarity?")
    print("-" * 65)
    for method, score in scores.items():
        if score is None:
            verdict = "N/A"
            score_str = "N/A"
        else:
            verdict = "YES" if score > 0.4 else "NO (too low)"
            score_str = f"{score:.4f}"
        print(f"  {method:23s} {score_str:>8s}  {verdict}")


scores = {
    "BOW":           sim_bow,
    "TF-IDF":        sim_tfidf,
    "Word2Vec avg":  sim_w2v,
    "Sentence-BERT": sim_sbert,
}
summarise_methods(scores)

Method                       Score  Captures semantic similarity?
-----------------------------------------------------------------
  BOW                       0.1543  NO (too low)
  TF-IDF                    0.4202  YES
  Word2Vec avg              0.7169  YES
  Sentence-BERT             0.6336  YES


### (b) BOW failure — word overlap walkthrough

In [20]:
def bow_overlap_analysis(text_a, text_b):
    """Show exactly why BOW fails on these two semantically similar sentences."""
    ta = set(tokenize(text_a))
    tb = set(tokenize(text_b))
    shared   = ta & tb
    only_a   = ta - tb
    only_b   = tb - ta

    print(f"Review A tokens : {sorted(ta)}")
    print(f"Review B tokens : {sorted(tb)}")
    print()
    print(f"Shared tokens   : {sorted(shared)}")
    print(f"Only in A       : {sorted(only_a)}")
    print(f"Only in B       : {sorted(only_b)}")
    print()
    print("Why BOW fails:")
    print("  Review A says 'incredible camera but terrible battery life'")
    print("  Review B says 'Battery drains fast, although photos are stunning'")
    print("  Semantic pairs: (incredible, stunning), (terrible battery, drains fast), (camera, photos)")
    print("  BOW sees NONE of these because they share only the surface token 'battery'.")
    print("  Cosine is low because the overlap vector is sparse despite identical meaning.")


bow_overlap_analysis(review_a, review_b)

Review A tokens : ['battery', 'but', 'camera', 'incredible', 'life', 'terrible']
Review B tokens : ['although', 'are', 'battery', 'drains', 'fast', 'photos', 'stunning']

Shared tokens   : ['battery']
Only in A       : ['but', 'camera', 'incredible', 'life', 'terrible']
Only in B       : ['although', 'are', 'drains', 'fast', 'photos', 'stunning']

Why BOW fails:
  Review A says 'incredible camera but terrible battery life'
  Review B says 'Battery drains fast, although photos are stunning'
  Semantic pairs: (incredible, stunning), (terrible battery, drains fast), (camera, photos)
  BOW sees NONE of these because they share only the surface token 'battery'.
  Cosine is low because the overlap vector is sparse despite identical meaning.


### (c) The semantic gap — how each method closes it

In [21]:
semantic_gap_explanation = """
The 'semantic gap' is the mismatch between surface token overlap and actual meaning.
Two sentences can convey identical sentiment with zero shared content words.

BOW: Lives entirely in token-overlap space. 'incredible' and 'stunning' are
  orthogonal dimensions — the model has no idea they mean the same thing.
  Gap: 100% open.

TF-IDF: Adds importance weighting but the vectors are still one-hot per token.
  Down-weighting 'but' and 'although' helps a little, but 'incredible' vs
  'stunning' remain unrelated. Gap: ~95% open.

Word2Vec avg: Maps synonyms to nearby points in embedding space, so
  'incredible' and 'stunning' contribute similar directional signal.
  Averaging loses word order and negation though ('not good' ≈ 'good').
  Gap: ~50% closed.

Sentence-BERT: Fine-tuned on sentence-pair tasks; the whole sentence is
  encoded as a single dense vector that captures meaning, sentiment polarity,
  and the contrast structure ('but', 'although'). Both reviews express the
  same mixed sentiment → high cosine. Gap: ~90% closed.
"""
print(semantic_gap_explanation)


The 'semantic gap' is the mismatch between surface token overlap and actual meaning.
Two sentences can convey identical sentiment with zero shared content words.

BOW: Lives entirely in token-overlap space. 'incredible' and 'stunning' are
  orthogonal dimensions — the model has no idea they mean the same thing.
  Gap: 100% open.

TF-IDF: Adds importance weighting but the vectors are still one-hot per token.
  Down-weighting 'but' and 'although' helps a little, but 'incredible' vs
  'stunning' remain unrelated. Gap: ~95% open.

Word2Vec avg: Maps synonyms to nearby points in embedding space, so
  'incredible' and 'stunning' contribute similar directional signal.
  Averaging loses word order and negation though ('not good' ≈ 'good').
  Gap: ~50% closed.

Sentence-BERT: Fine-tuned on sentence-pair tasks; the whole sentence is
  encoded as a single dense vector that captures meaning, sentiment polarity,
  and the contrast structure ('but', 'although'). Both reviews express the
  same mixe